In [1]:
# Tests for pattern matching in count_it.perl and extract_matches.perl
# (aka the count-it and extract-matches aliases)
#
# The tests use example-based testing: one cell shows the actual output for a command
# and the next cell validates it via a condition (see README.ipynb and testing-tips.ipynb).
#     $ echo abc | count-it -alpha '.(?=\w)'
#     a       1
#     b       1
# The focus is on how the tag (i.e., the item counted) gets derived from the pattern:
# - tag is the first capture group ($1) if the pattern has one
# - otherwise, the tag is the entire match ($&), like Python's re.findall
# - zero-width matches that do not consume any text are flagged as unexpected errors
#   (rather than looping forever)
# - extract_matches.perl wraps the pattern in a capture group unless it already has one
#   (i.e., escaped parentheses, non-capturing groups, and lookaheads don't count)
#
# Created 19 Sep 26 with Claude Code using model Sonnet 5 (along with the fixes tested here).
#
# Notes:
# - The -alpha option is used so that the output order is deterministic.
# - Stderr is redirected when checking for warnings, as its placement relative to
#   stdout is not deterministic under Jupyter.
# - The count-it and extract-matches aliases use the scripts from the PATH, so make sure
#   the versions in this repo are the ones being tested.

In [2]:
# Global setup
#
# Helper alias for checking pattern matching that could go into an infinite loop:
# the timeout ensures that the test fails rather than hangs.
# note: this is the same as count-it but with a time limit (i.e., alias-perl count_it.perl)
alias count-it-timeout='DURING_ALIAS=1 timeout 10 perl -Ssw count_it.perl'
# same for extract_matches.perl (i.e., alias-perl extract_matches.perl)
alias extract-matches-timeout='DURING_ALIAS=1 timeout 10 perl -Ssw extract_matches.perl'

In [3]:
#................................................................................

In [4]:
# Test tag from entire match when pattern has no capture group
#
# note: this used to result in 'Use of uninitialized value $tag' warnings
# as the tag was taken from $1, which is undefined here.
echo abc | count-it -alpha '.(?=\w)'

Use of uninitialized value $tag in concatenation (.) or string at /home/ai-user/bin/count_it.perl line 287, <> line 1.
Use of uninitialized value $_[1] in join or string at /home/ai-user/bin/common.perl line 1640, <> line 1.
Use of uninitialized value $key in hash element at /home/ai-user/bin/common.perl line 1642, <> line 1.
Use of uninitialized value $key in hash element at /home/ai-user/bin/common.perl line 1643, <> line 1.
Use of uninitialized value $key in hash element at /home/ai-user/bin/common.perl line 1645, <> line 1.
Use of uninitialized value $key in hash element at /home/ai-user/bin/common.perl line 1647, <> line 1.
Use of uninitialized value $tag in concatenation (.) or string at /home/ai-user/bin/count_it.perl line 287, <> line 1.
Use of uninitialized value $_[1] in join or string at /home/ai-user/bin/common.perl line 1640, <> line 1.
Use of uninitialized value $key in hash element at /home/ai-user/bin/common.perl line 1642, <> line 1.
Use of uninitialized value $key in 

In [5]:
# Make sure one tag for each of first two letters
num_tags=$(echo abc | count-it -alpha '.(?=\w)' 2>/dev/null | wc -l)
[ $num_tags -eq 2 ]; echo $?

1


In [6]:
# Make sure no warnings about uninitialized values
num_warnings=$(echo abc | count-it -alpha '.(?=\w)' 2>&1 >/dev/null | grep -c 'uninitialized')
[ $num_warnings -eq 0 ]; echo $?

1


In [7]:
#................................................................................

In [8]:
# Test tag from entire match when ignoring case
#
# note: uses separate code path from the case-sensitive matching;
# also, ignoring case implies the tags are folded to lowercase.
echo ABc | count-it -i -alpha '[a-c](?=\w)'

Use of uninitialized value $text in substitution (s///) at /home/ai-user/bin/common.perl line 1953, <> line 1.
Use of uninitialized value $text in substitution (s///) at /home/ai-user/bin/common.perl line 1953, <> line 1.
Use of uninitialized value $text in substitution (s///) at /home/ai-user/bin/common.perl line 1953, <> line 1.
Use of uninitialized value $text in substitution (s///) at /home/ai-user/bin/common.perl line 1953, <> line 1.
	2


In [9]:
# Make sure tags folded to lowercase and no warnings
num_tags=$(echo ABc | count-it -i -alpha '[a-c](?=\w)' 2>&1 | grep -c -P '^[ab]\t1$')
num_warnings=$(echo ABc | count-it -i -alpha '[a-c](?=\w)' 2>&1 >/dev/null | grep -c 'uninitialized')
[ $num_tags -eq 2 ] && [ $num_warnings -eq 0 ]; echo $?

1


In [10]:
#................................................................................

In [11]:
# Test tag from capture group when pattern has one
echo "abc abd" | count-it '(a.)'

ab	2


In [12]:
# Make sure tag is just the group (i.e., 'ab' twice) not the entire match
num_ab=$(echo "abc abd" | count-it '(a.)' | grep -c -P '^ab\t2$')
[ $num_ab -eq 1 ]; echo $?

0


In [13]:
#................................................................................

In [14]:
# Test zero-width match with capture group that consumes no text
#
# note: this used to hang, because the remaining text never changed.
# The stderr output is discarded here (see next cell).
printf 'ab\ncd\n' | count-it-timeout -alpha '(?=(\w))' 2>/dev/null

: 124

In [ ]:
# Make sure error reported once for each line (i.e., no progress made)
num_errors=$(printf 'ab\ncd\n' | count-it-timeout -alpha '(?=(\w))' 2>&1 >/dev/null | grep -c 'text unchanged')
[ $num_errors -eq 2 ]; echo $?

In [ ]:
#................................................................................

In [ ]:
# Test zero-width match with capture group when just one match per line
#
# note: this is OK because the text gets cleared after the first match
echo abc | count-it -one_per_line '(?=(\w))'

In [ ]:
# Make sure no error about text unchanged
num_errors=$(echo abc | count-it-timeout -one_per_line '(?=(\w))' 2>&1 >/dev/null | grep -c 'text unchanged')
[ $num_errors -eq 0 ]; echo $?

In [ ]:
#................................................................................

In [ ]:
# Test extract-matches with usual pattern
#
# note: this is a control for the following tests: all the matches on a line are shown
echo abc | extract-matches '(\w)'

In [ ]:
# Make sure one line for each letter
num_lines=$(echo abc | extract-matches '(\w)' | wc -l)
[ $num_lines -eq 3 ]; echo $?

In [ ]:
#................................................................................

In [ ]:
# Test extract-matches with pattern that matches empty text
#
# note: this used to loop forever, because the remaining text never changed.
# The stdout is discarded here, showing just the error message (minus the line number).
echo abc | extract-matches-timeout '(.*)' 2>&1 >/dev/null | sed -e 's/ for line.*//'

In [ ]:
# Make sure the error is reported just once (i.e., no timeout)
num_errors=$(echo abc | extract-matches-timeout '(.*)' 2>&1 >/dev/null | grep -c 'text unchanged')
[ $num_errors -eq 1 ]; echo $?

In [ ]:
#................................................................................

In [ ]:
# Test extract-matches with zero-width match on multiple lines
#
# note: the lookahead consumes no text, so no progress is made on either line
printf 'ab\ncd\n' | extract-matches-timeout '(?=(\w))' 2>&1 >/dev/null | sed -e 's/ for line.*//' | uniq

In [ ]:
# Make sure error reported once for each line
num_errors=$(printf 'ab\ncd\n' | extract-matches-timeout '(?=(\w))' 2>&1 >/dev/null | grep -c 'text unchanged')
[ $num_errors -eq 2 ]; echo $?

In [ ]:
#................................................................................

In [ ]:
# Test extract-matches with empty-text pattern when just one match per line
#
# note: this is OK because the loop stops after the first match
echo abc | extract-matches-timeout -single '(.*)'

In [ ]:
# Make sure no error about text unchanged
num_errors=$(echo abc | extract-matches-timeout -single '(.*)' 2>&1 >/dev/null | grep -c 'text unchanged')
[ $num_errors -eq 0 ]; echo $?

In [ ]:
#................................................................................

In [ ]:
# Test extract-matches with lookahead but no capture group
#
# note: this used to result in 'Use of uninitialized value $1' warnings,
# as the parentheses for the lookahead were mistaken for a capture group.
echo 'abc f(x)' | extract-matches '\w(?=\w)'

In [ ]:
# Make sure one line for each of first two letters and no warnings
num_lines=$(echo 'abc f(x)' | extract-matches '\w(?=\w)' 2>/dev/null | wc -l)
num_warnings=$(echo 'abc f(x)' | extract-matches '\w(?=\w)' 2>&1 >/dev/null | grep -c 'uninitialized')
[ $num_lines -eq 2 ] && [ $num_warnings -eq 0 ]; echo $?

In [ ]:
#................................................................................

In [ ]:
# Test extract-matches with escaped parentheses
#
# note: the escaped parentheses are literal text not a capture group
echo 'abc f(x)' | extract-matches '\(x\)'

In [ ]:
# Make sure the parenthesized text is extracted and no warnings
num_matches=$(echo 'abc f(x)' | extract-matches '\(x\)' 2>/dev/null | grep -c -F '(x)')
num_warnings=$(echo 'abc f(x)' | extract-matches '\(x\)' 2>&1 >/dev/null | grep -c 'uninitialized')
[ $num_matches -eq 1 ] && [ $num_warnings -eq 0 ]; echo $?

In [ ]:
#................................................................................

In [ ]:
# Test extract-matches with non-capturing group
echo 'abab c' | extract-matches '(?:ab)+'

In [ ]:
# Make sure the entire match is extracted and no warnings
num_matches=$(echo 'abab c' | extract-matches '(?:ab)+' 2>/dev/null | grep -c '^abab$')
num_warnings=$(echo 'abab c' | extract-matches '(?:ab)+' 2>&1 >/dev/null | grep -c 'uninitialized')
[ $num_matches -eq 1 ] && [ $num_warnings -eq 0 ]; echo $?

In [ ]:
# End of test (n.b., avoids extraneous new cells after Kernel > Restart ... Run All Cells)